In [10]:
import pandas as pd
import numpy as np
import re
from urllib.parse import urlparse
import warnings
warnings.filterwarnings("ignore")
print("✅ Imports done!")

✅ Imports done!


In [11]:
df = pd.read_csv("../data/raw_urls.csv")
print(f"Shape: {df.shape}")
df.head()

Shape: (641120, 2)


,url,label
0,br-icloud.com.br,1
1,mp3raid.com/music/krizz_kaliko.html,0
2,bopsecrets.org/rexroth/cr/1.htm,0
3,http://www.garage-pirenne.be/index.php?option=...,1
4,http://adventure-nicaragua.net/index.php?optio...,1


In [12]:
import re
from urllib.parse import urlparse

BRAND_DOMAINS = {
    "paypal": ["paypal.com"],
    "amazon": ["amazon.com", "amazon.co.uk", "amazon.de"],
    "apple": ["apple.com", "icloud.com"],
    "microsoft": ["microsoft.com", "live.com", "office.com", "outlook.com"],
    "google": ["google.com", "gmail.com", "youtube.com"],
    "facebook": ["facebook.com", "fb.com"],
    "instagram": ["instagram.com"],
    "netflix": ["netflix.com"],
    "ebay": ["ebay.com"],
    "chase": ["chase.com"],
    "wellsfargo": ["wellsfargo.com"],
    "bankofamerica": ["bankofamerica.com"],
}

def has_brand_mismatch(url, netloc):
    url_lower = url.lower()
    netloc_lower = netloc.lower()

    for brand, official_domains in BRAND_DOMAINS.items():
        if brand in url_lower:
            is_legit = any(
                netloc_lower == d or netloc_lower.endswith("." + d)
                for d in official_domains
            )
            if not is_legit:
                return 1
    return 0

def extract_features(url):
    features = {}
    try:
        parsed = urlparse(url if url.startswith("http") else "http://" + url)
        features["url_length"] = len(url)
        features["has_https"] = 1 if parsed.scheme == "https" else 0
        features["num_dots"] = url.count(".")
        features["num_hyphens"] = url.count("-")
        features["num_underscores"] = url.count("_")
        features["num_slashes"] = url.count("/")
        features["num_question"] = url.count("?")
        features["num_equals"] = url.count("=")
        features["num_at"] = url.count("@")
        features["num_ampersand"] = url.count("&")
        features["num_exclamation"] = url.count("!")
        features["num_percent"] = url.count("%")
        features["num_digits"] = sum(c.isdigit() for c in url)
        features["num_special_chars"] = sum(not c.isalnum() for c in url)

        domain = parsed.netloc
        features["domain_length"] = len(domain)
        features["num_subdomains"] = domain.count(".")
        features["has_ip"] = 1 if re.match(r'\d+\.\d+\.\d+\.\d+', domain) else 0
        features["has_hyphen_domain"] = 1 if "-" in domain else 0

        path = parsed.path
        features["path_length"] = len(path)
        features["num_path_segments"] = path.count("/")
        features["has_php"] = 1 if ".php" in url else 0
        features["has_html"] = 1 if ".html" in url else 0
        features["has_exe"] = 1 if ".exe" in url else 0
        features["has_zip"] = 1 if ".zip" in url else 0

        # brand names removed from here — now handled by has_brand_mismatch instead,
        # so a brand's own legitimate domain no longer gets penalized
        sus_words = [
            "login","secure","verify","account","update",
            "confirm","banking","password","signin",
            "free","lucky","winner","click","alert"
        ]

        url_lower = url.lower()

        features["suspicious_words"] = sum(1 for w in sus_words if w in url_lower)
        features["has_login"] = 1 if "login" in url_lower else 0
        features["has_secure"] = 1 if "secure" in url_lower else 0
        features["has_verify"] = 1 if "verify" in url_lower else 0
        features["has_account"] = 1 if "account" in url_lower else 0
        features["has_free"] = 1 if "free" in url_lower else 0

        suspicious_tlds = [
            ".ru",".cn",".tk",".ml",".ga",".cf",
            ".gq",".xyz",".top",".click",".download",
            ".zip",".review"
        ]

        features["suspicious_tld"] = 1 if any(url_lower.endswith(t) for t in suspicious_tlds) else 0
        features["digit_ratio"] = features["num_digits"]/len(url) if len(url)>0 else 0
        features["special_ratio"] = features["num_special_chars"]/len(url) if len(url)>0 else 0

        # new 34th feature
        features["has_brand_mismatch"] = has_brand_mismatch(url, domain)

    except Exception:
        for key in features:
            features[key] = 0

    return features

print("✅ Feature function defined!")
print("Number of features:", len(extract_features("http://test.com")))
print(extract_features("http://test.com").keys())

✅ Feature function defined!
Number of features: 34
dict_keys(['url_length', 'has_https', 'num_dots', 'num_hyphens', 'num_underscores', 'num_slashes', 'num_question', 'num_equals', 'num_at', 'num_ampersand', 'num_exclamation', 'num_percent', 'num_digits', 'num_special_chars', 'domain_length', 'num_subdomains', 'has_ip', 'has_hyphen_domain', 'path_length', 'num_path_segments', 'has_php', 'has_html', 'has_exe', 'has_zip', 'suspicious_words', 'has_login', 'has_secure', 'has_verify', 'has_account', 'has_free', 'suspicious_tld', 'digit_ratio', 'special_ratio', 'has_brand_mismatch'])


In [13]:
print("Extracting features from all URLs...")
print("This will take 2-3 minutes...")
features_list = []
for i, url in enumerate(df["url"]):
    features_list.append(extract_features(str(url)))
    if (i+1) % 50000 == 0:
        print(f"Processed {i+1:,} / {len(df):,} URLs...")
df_features = pd.DataFrame(features_list)
df_features["label"] = df["label"].values
print(f"\n✅ Features extracted! Shape: {df_features.shape}")
df_features.head()

Extracting features from all URLs...
This will take 2-3 minutes...
Processed 50,000 / 641,120 URLs...
Processed 100,000 / 641,120 URLs...
Processed 150,000 / 641,120 URLs...
Processed 200,000 / 641,120 URLs...
Processed 250,000 / 641,120 URLs...
Processed 300,000 / 641,120 URLs...
Processed 350,000 / 641,120 URLs...
Processed 400,000 / 641,120 URLs...
Processed 450,000 / 641,120 URLs...
Processed 500,000 / 641,120 URLs...
Processed 550,000 / 641,120 URLs...
Processed 600,000 / 641,120 URLs...

✅ Features extracted! Shape: (641120, 35)


,url_length,has_https,num_dots,num_hyphens,num_underscores,num_slashes,num_question,num_equals,num_at,num_ampersand,...,has_login,has_secure,has_verify,has_account,has_free,suspicious_tld,digit_ratio,special_ratio,has_brand_mismatch,label
0,16.0,0.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.187500,0.0,1
1,35.0,0.0,2.0,0.0,1.0,2.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.028571,0.142857,0.0,0
2,31.0,0.0,2.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.032258,0.161290,0.0,0
3,88.0,0.0,3.0,1.0,2.0,3.0,1.0,4.0,0.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.079545,0.204545,0.0,1
4,235.0,0.0,2.0,1.0,1.0,3.0,1.0,3.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.093617,0.059574,0.0,1


In [14]:
df_features.to_csv(
    r"C:\Users\Jyoti Hanagandi\OneDrive\Desktop\cyber-threat-detection\data\features.csv",
    index=False
)

print("✅ STEP 2 COMPLETE!")
print("Shape:", df_features.shape)

✅ STEP 2 COMPLETE!
Shape: (641120, 35)
